In [0]:
display(dbutils.fs.ls("/Volumes/workspace/olist/raw"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/olist/raw/olist_customers_dataset.csv,olist_customers_dataset.csv,9033957,1793179819000
dbfs:/Volumes/workspace/olist/raw/olist_geolocation_dataset.csv,olist_geolocation_dataset.csv,61273883,1793179867000
dbfs:/Volumes/workspace/olist/raw/olist_order_items_dataset.csv,olist_order_items_dataset.csv,15438671,1793179833000
dbfs:/Volumes/workspace/olist/raw/olist_order_payments_dataset.csv,olist_order_payments_dataset.csv,5777138,1793179813000
dbfs:/Volumes/workspace/olist/raw/olist_order_reviews_dataset.csv,olist_order_reviews_dataset.csv,14451670,1793179831000
dbfs:/Volumes/workspace/olist/raw/olist_orders_dataset.csv,olist_orders_dataset.csv,17654914,1793179835000
dbfs:/Volumes/workspace/olist/raw/olist_products_dataset.csv,olist_products_dataset.csv,2379446,1793179801000
dbfs:/Volumes/workspace/olist/raw/olist_sellers_dataset.csv,olist_sellers_dataset.csv,174703,1793179788000
dbfs:/Volumes/workspace/olist/raw/product_category_name_translation.csv,product_category_name_translation.csv,2613,1793179787000


In [0]:
from pyspark.sql import functions as F

RAW = "/Volumes/workspace/olist/raw"
files = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

for name, fname in files.items():
    df = (spark.read
          .option("header", True)
          .option("inferSchema", True)
          .option("multiLine", True)
          .option("escape", '"')
          .csv(f"{RAW}/{fname}")
          .withColumn("ingested_at", F.current_timestamp()))
    df.write.mode("overwrite").format("delta").saveAsTable(f"workspace.olist.bronze_{name}")
    print(f"bronze_{name}: {df.count()} rows")

bronze_orders: 99441 rows
bronze_order_items: 112650 rows
bronze_customers: 99441 rows
bronze_order_payments: 103886 rows
bronze_order_reviews: 99224 rows
bronze_products: 32951 rows
bronze_sellers: 3095 rows
bronze_geolocation: 1000163 rows
bronze_category_translation: 71 rows
